###hetha serveur responsable a l'entrainement du modele a chaque fois avec les données

Enregistrement de data processed


In [ ]:
################################

In [5]:
######################################Le dernier script 
##############
#le script  post get image 
import os
import json
import pymongo
import cv2
import pytesseract
import shutil
from datetime import datetime
from bson import ObjectId
from pathlib import Path
from tqdm import tqdm
import warnings
import random
from PIL import Image
import pandas as pd
from pathlib import Path
import numpy as np
from datasets import Dataset 
from datasets import load_from_disk
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
from transformers import AutoTokenizer
import os
from transformers import AutoModelForTokenClassification, AutoProcessor
warnings.filterwarnings("ignore")
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from transformers import TrainingArguments
import evaluate
from transformers import Trainer, default_data_collator
from datasets import Features, Sequence, Value, Array2D, Array3D
import re
import requests  # Import manquant essentiel
from requests.exceptions import RequestException
#
labels_list = ['O', 'B-COMPANY', 'B-DATE', 'B-ADDRESS', 'B-TOTAL', 'B-INVOICE_NO', 'I-INVOICE_NO',
    'B-ITEM_DESC', 'I-ITEM_DESC',
    'B-ITEM_QTY', 'I-ITEM_QTY',
    'B-ITEM_PRICE', 'I-ITEM_PRICE']
ids2labels = {k: v for k, v in enumerate(labels_list)}
labels2ids = {v: k for k, v in enumerate(labels_list)}
#model = AutoModelForTokenClassification.from_pretrained(model_id, label2id=labels2ids, id2label=ids2labels)

# === Configuration Globale ===
CONFIG = {
    "mongo": {
        "host": "localhost",
        "port": 27017,
        "db": "Telnet",
        "collections": {
            "invoices": "invoices",
            "images": "imageinvoices"
        }
    },
     "node_api": {
        "base_url": "http://192.168.1.158:8000",  # serveur Node.js
        "image_endpoint": "/api/image/{filename}",  # Endpoint pour récupérer une image par filename
        "timeout": 30,
        "max_retries": 3
    }
}

# === Fonction Cellule 1 : Initialisation ===
def init_environment(base_folder='C:/Users/HP/Desktop/sequencedemodel'):
    """Initialise l'environnement et crée le dossier de sortie"""
    os.environ['IPYTHONDIR'] = os.path.join(os.path.expanduser('~'), '.ipython_no_checkpoints')
    today = datetime.now().strftime("%d-%m-%Y")
    output_folder = os.path.join(base_folder, today)
    os.makedirs(output_folder, exist_ok=True)
    return output_folder

# === Fonction Cellule 2 : Export des factures ===
def export_invoices(output_folder):
    """Exporte les factures depuis MongoDB vers des fichiers texte"""
    class DateTimeEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, (datetime, ObjectId)):
                return str(obj)
            return super().default(obj)

    output_dir = os.path.join(output_folder, "entities")
    os.makedirs(output_dir, exist_ok=True)

    try:
        client = pymongo.MongoClient(
            host=CONFIG["mongo"]["host"],
            port=CONFIG["mongo"]["port"]
        )
        collection = client[CONFIG["mongo"]["db"]][CONFIG["mongo"]["collections"]["invoices"]]

        for doc in collection.find():
            try:
                doc_id = str(doc["_id"])
                filepath = os.path.join(output_dir, f"{doc_id}.txt")
                
                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(doc, f, cls=DateTimeEncoder, indent=2)
                
            except Exception as e:
                print(f"Erreur avec le document {doc.get('_id', 'inconnu')}: {str(e)}")

    except Exception as e:
        print(f"Erreur MongoDB: {str(e)}")
    finally:
        if 'client' in locals():
            client.close()
###############
def download_image_from_node(image_url, save_path):
    """Version basique sans gestion d'erreur complexe"""
    try:
        response = requests.get(image_url, timeout=10)
        response.raise_for_status()  # Vérifie juste que le statut HTTP est 200
        
        with open(save_path, 'wb') as f:
            f.write(response.content)
        return True
        
    except Exception as e:
        print(f"Échec du téléchargement pour {save_path}: {str(e)}")
        return False
# === Fonction Cellule 3 : Export des images ===
def export_images(output_folder, node_api_url="http://localhost:8000"):
    """Exporte les images depuis l'API Node.js en conservant la logique de nommage originale"""
    img_dir = os.path.join(output_folder, "img")
    os.makedirs(img_dir, exist_ok=True)

    try:
        client = pymongo.MongoClient(
            host=CONFIG["mongo"]["host"],
            port=CONFIG["mongo"]["port"]
        )
        db = client[CONFIG["mongo"]["db"]]
        images_col = db[CONFIG["mongo"]["collections"]["images"]]
        invoices_col = db[CONFIG["mongo"]["collections"]["invoices"]]

        # 1. Création du mapping image_id -> filename (identique à l'original)
        images_map = {str(img["_id"]): img["filename"] for img in images_col.find()}

        # 2. Export des images avec la même logique de nommage
        for invoice in invoices_col.find({"image": {"$ne": None}}):
            try:
                invoice_id = str(invoice["_id"])
                image_id = str(invoice["image"])
                
                if image_id in images_map:
                    # Construction du nom de fichier final (identique à l'original)
                    dest_filename = f"{invoice_id}.jpg"
                    dest_path = os.path.join(img_dir, dest_filename)
                    
                    # Téléchargement via API (nouveauté)
                    original_filename = images_map[image_id]
                    image_url = f"{node_api_url}/api/images/{original_filename}"
                    #image_url = f"{NODE_API_URL}/api/images/{original_filename}"
                    if download_image_from_node(image_url, dest_path):
                        print(f"Image sauvegardée: {dest_filename} (original: {original_filename})")
                    else:
                        print(f"Échec du téléchargement pour {dest_filename}")

            except Exception as e:
                print(f"Erreur avec la facture {invoice_id}: {str(e)}")

    except Exception as e:
        print(f"ERREUR CRITIQUE: {str(e)}")
    finally:
        if 'client' in locals():
            client.close()
#____fonction ocr te3i

def extract_text_with_boxes(image_path, output_file):
     # Afficher les chemins absolus dès le début
    abs_input = os.path.abspath(input_folder)
    abs_output = os.path.abspath(output_folder)
    print(f"\nChemin absolu du dossier d'images: {abs_input}")
    print(f"Chemin absolu du dossier de sortie: {abs_output}\n")
    # Chargement et prétraitement de l'image
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Amélioration du contraste
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    # Configuration pour Tesseract
    custom_config = r'--oem 3 --psm 11 -l eng+fra'
    
    # Extraction des données avec Tesseract
    data = pytesseract.image_to_data(gray, config=custom_config, output_type=pytesseract.Output.DICT)
    
    # Organisation des résultats par blocs et lignes
    block_line_words = {}
    for i in range(len(data['text'])):
        if data['text'][i].strip():
            block_num = data['block_num'][i]
            line_num = data['line_num'][i]
            
            if (block_num, line_num) not in block_line_words:
                block_line_words[(block_num, line_num)] = []
                
            # Coordonnées du rectangle
            x = data['left'][i]
            y = data['top'][i]
            w = data['width'][i]
            h = data['height'][i]
            
            block_line_words[(block_num, line_num)].append({
                'text': data['text'][i],
                'x': x,
                'y': y,
                'w': w,
                'h': h,
                'conf': data['conf'][i]
            })
    
    # Regroupement des mots en entités
    results = []
    
    for (block, line), words in block_line_words.items():
        words.sort(key=lambda w: w['x'])
        current_group = []
        
        for i, word in enumerate(words):
            current_group.append(word)
            
            should_merge = False
            
            if i < len(words) - 1:
                next_word = words[i + 1]
                distance = next_word['x'] - (word['x'] + word['w'])
                avg_height = (word['h'] + next_word['h']) / 2
                should_merge = distance < avg_height * 0.8
                
                current_text = ''.join([w['text'] for w in current_group])
                next_text = next_word['text']
                
                date_pattern = re.search(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}$', current_text)
                if date_pattern and re.match(r'\d{1,2}:\d{1,2}(:\d{1,2})?(\s*[AP]M)?', next_text):
                    should_merge = True
                
                address_indicators = ['JALAN', 'JLN', 'NO.', 'NO ', 'BANDAR', 'TEL', 'FAX', 'EMAIL']
                if any(indicator in current_text for indicator in address_indicators):
                    should_merge = True
                
                company_indicators = ['SDN', 'BHD', 'PTE', 'LTD', 'MARKETING', 'GROUP', 'COMPANY']
                if any(indicator in current_text for indicator in company_indicators) and any(indicator in next_text for indicator in company_indicators):
                    should_merge = True
            
            if i == len(words) - 1 or not should_merge:
                if current_group:
                    min_x = min(w['x'] for w in current_group)
                    min_y = min(w['y'] for w in current_group)
                    max_x = max(w['x'] + w['w'] for w in current_group)
                    max_y = max(w['y'] + w['h'] for w in current_group)
                    
                    box = [
                        [min_x, min_y],
                        [max_x, min_y],
                        [max_x, max_y],
                        [min_x, max_y]
                    ]
                    
                    text = ' '.join(w['text'] for w in current_group)
                    results.append({'box': box, 'text': text})
                    current_group = []
    
    # Écrire les résultats dans le fichier de sortie
    with open(output_file, 'w', encoding='utf-8') as f:
        for result in results:
            box = result['box']
            text = result['text']
            line = f"{box[0][0]},{box[0][1]},{box[1][0]},{box[1][1]},{box[2][0]},{box[2][1]},{box[3][0]},{box[3][1]},{text}\n"
            f.write(line)
    
    return results

def process_images_folder(input_folder, output_folder):
    """
    Traite toutes les images du dossier d'entrée et sauvegarde les résultats dans le dossier de sortie
    
    Args:
        input_folder (str): Chemin vers le dossier contenant les images
        output_folder (str): Chemin vers le dossier où sauvegarder les résultats
    """
    # Créer le dossier de sortie s'il n'existe pas
    os.makedirs(output_folder, exist_ok=True)
    
    # Lister tous les fichiers image dans le dossier d'entrée
    image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp']
    image_files = [f for f in os.listdir(input_folder) 
                  if os.path.splitext(f)[1].lower() in image_extensions]
    
    # Traiter chaque image avec une barre de progression
    for filename in tqdm(image_files, desc="Processing images"):
        input_path = os.path.join(input_folder, filename)
        output_filename = os.path.splitext(filename)[0] + '.txt'
        output_path = os.path.join(output_folder, output_filename)
        
        try:
            extract_text_with_boxes(input_path, output_path)
        except Exception as e:
            print(f"Erreur lors du traitement de {filename}: {str(e)}")

# === Fonction Cellule 4 : Traitement OCR ===
def process_images_with_ocr(output_folder):
    """Applique l'OCR sur les images exportées"""
    input_img_dir = os.path.join(output_folder, "img")
    output_box_dir = os.path.join(output_folder, "box")
    os.makedirs(output_box_dir, exist_ok=True)

    def extract_text(image_path, output_file):
        img = cv2.imread(image_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)
        
        custom_config = r'--oem 3 --psm 11 -l eng+fra'
        data = pytesseract.image_to_data(gray, config=custom_config, output_type=pytesseract.Output.DICT)
        
        with open(output_file, 'w', encoding='utf-8') as f:
            for i in range(len(data['text'])):
                if data['text'][i].strip():
                    line = f"{data['left'][i]},{data['top'][i]},{data['width'][i]},{data['height'][i]},{data['text'][i]}\n"
                    f.write(line)

    # Traitement de toutes les images
    for img_file in tqdm(os.listdir(input_img_dir)):
        if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
            input_path = os.path.join(input_img_dir, img_file)
            output_path = os.path.join(output_box_dir, f"{os.path.splitext(img_file)[0]}.txt")
            
            try:
                extract_text(input_path, output_path)
            except Exception as e:
                print(f"Erreur avec {img_file}: {str(e)}")
#fc
def clean_orphaned_entities(output_folder):
    """
    Supprime les fichiers texte dans 'entities' sans image correspondante dans 'img'
    
    Args:
        output_folder (str): Chemin du dossier racine contenant les sous-dossiers 'entities' et 'img'
    
    Returns:
        int: Nombre de fichiers supprimés
    """
    entities_dir = os.path.join(output_folder, "entities")
    img_dir = os.path.join(output_folder, "img")
    
    # Vérification des dossiers
    if not os.path.exists(entities_dir):
        raise FileNotFoundError(f"Dossier 'entities' introuvable : {entities_dir}")
    if not os.path.exists(img_dir):
        raise FileNotFoundError(f"Dossier 'img' introuvable : {img_dir}")

    deleted_count = 0
    img_extensions = ('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG')
    
    for filename in os.listdir(entities_dir):
        if filename.endswith(".txt"):
            base_name = os.path.splitext(filename)[0]
            img_found = False
            
            # Vérifier toutes les extensions d'image possibles
            for ext in img_extensions:
                if os.path.exists(os.path.join(img_dir, base_name + ext)):
                    img_found = True
                    break
            
            # Suppression si image non trouvée
            if not img_found:
                txt_path = os.path.join(entities_dir, filename)
                os.remove(txt_path)
                deleted_count += 1
                print(f"Supprimé : {filename} (image manquante)")
    
    print(f"Nettoyage terminé. Fichiers supprimés : {deleted_count}")
    return deleted_count
#fc
from difflib import SequenceMatcher
#la methode 1 avant la normalisation 
#la meilleur methode 
def get_word_tag(word, entities):
  #Assure la cohérence majuscules/minuscules"""
    word_lower = word.lower()

    # 1. D'abord vérifier les entités principales (même logique que l'original)
    for entity in entities:
        entity_upper = entity.upper()  # Normalisation en majuscules
        entity_value = str(entities[entity]).lower()

        # Logique originale inchangée
        if SequenceMatcher(None, word_lower, entity_value).ratio() >= 0.8:
            return labels2ids[f'B-{entity_upper}']
        elif entity_upper in ['ADDRESS', 'COMPANY'] and word_lower in entity_value:
            return labels2ids[f'B-{entity_upper}']
        elif entity_upper in ['DATE', 'TOTAL'] and entity_value in word_lower:
            return labels2ids[f'B-{entity_upper}']

    # 2. Vérification spéciale pour les items (nouveauté)
    if 'items' in entities:
        for item in entities['items']:
            # Description
            desc = str(item.get('description', '')).lower()
            if SequenceMatcher(None, word_lower, desc).ratio() >= 0.7:
                return labels2ids['B-ITEM_DESC']

            # Quantité
            qty = str(item.get('quantity', '')).lower()
            if word_lower == qty:
                return labels2ids['B-ITEM_QTY']

            # Prix
            price = str(item.get('unit_price', '')).lower()
            if word_lower in price.replace(" ", ""):
                return labels2ids['B-ITEM_PRICE']

    return labels2ids['O']
#fc
def get_normalized_bbox(line, img_width, img_height):
  line = line.strip().split(',')
  x1, y1, x2, y2 = int(line[0]), int(line[1]), int(line[6]), int(line[7])
  return [x1 / img_width * 1000, y1 / img_height * 1000, x2 / img_width * 1000, y2 / img_height * 1000]
#fc
import json


#
def get_word(line):
  line = line.strip().split(',')
  return ','.join(line[8:])

#fc
def get_entities_dict(entities_file_path):

    def safe_str(value):
        """Convertit une valeur en string vide si None"""
        return str(value) if value is not None else ""

    with open(entities_file_path, 'r', encoding='utf-8') as entities_file:
        entities = json.load(entities_file)

    res_dict = {
        'COMPANY': safe_str(entities.get('company')),
        'DATE': safe_str(entities.get('date')),
        'ADDRESS': safe_str(entities.get('address')),
        'TOTAL': safe_str(entities.get('total')),
        'invoice_no': safe_str(entities.get('invoice_no')),
        'items': []
    }

    # Traitement des items
    for item in entities.get('items', []):
        res_dict['items'].append({
            'description': safe_str(item.get('description')),
            'quantity': safe_str(item.get('quantity')),
            'unit_price': safe_str(item.get('unit_price'))
        })

    return res_dict

#fc 
def get_image_dimensions(img_path):
  img = Image.open(img_path)
  return img.size
#data_dir bich ye5ou unprocessed data folder bich imappy test w train
def create_df(data_dir):
    box_path = os.path.join(data_dir, 'box')
    img_path = os.path.join(data_dir, 'img')
    entities_path = os.path.join(data_dir, 'entities')
    
    # Vérification que les dossiers existent
    if not all(os.path.exists(p) for p in [box_path, img_path, entities_path]):
        print(f"❌ Dossiers manquants dans {data_dir}")
        return pd.DataFrame()
    
    print(f"Chemins utilisés :\n  📦 Box : {box_path}\n  🖼️ Image : {img_path}\n  🧠 Entities : {entities_path}")
    
    image_path_list = []
    bboxes_list = []
    words_list = []
    ner_tags_list = []
    error_count = 0

    # Parcourir les fichiers BOX qui déterminent les documents disponibles
    for box_file in os.listdir(box_path):
        if box_file.endswith('.txt'):
            id = Path(box_file).stem
            img_file = None
            
            # Trouver le fichier image correspondant (supporte plusieurs extensions)
            for ext in ['.png', '.jpg', '.jpeg']:
                potential_img = os.path.join(img_path, f"{id}{ext}")
                if os.path.exists(potential_img):
                    img_file = potential_img
                    break
            
            if not img_file:
                print(f"⚠️ Image manquante pour {id}")
                error_count += 1
                continue
                
            entities_file = os.path.join(entities_path, f"{id}.txt")
            if not os.path.exists(entities_file):
                print(f"⚠️ Entities manquant pour {id}")
                error_count += 1
                continue

            try:
                # Traitement du fichier
                img_width, img_height = get_image_dimensions(img_file)
                
                with open(os.path.join(box_path, box_file), 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    words = [get_word(line) for line in lines]
                    bboxes = [get_normalized_bbox(line, img_width, img_height) for line in lines]

                entities = get_entities_dict(entities_file)
                ner_tags = [get_word_tag(word, entities) for word in words]

                image_path_list.append(img_file)
                bboxes_list.append(bboxes)
                words_list.append(words)
                ner_tags_list.append(ner_tags)
                
            except Exception as e:
                print(f"❌ Erreur avec {id}: {str(e)}")
                error_count += 1

    print(f"Documents traités avec succès: {len(image_path_list)}, erreurs: {error_count}")
    
    return pd.DataFrame({
        'image_path': image_path_list,
        'bboxes': bboxes_list,
        'words': words_list,
        'ner_tags': ner_tags_list
    })
def create_stats_df(df):
    """
    Version finale 100% compatible avec vos besoins :
    - Prend en compte tous vos labels exacts
    - Conserve votre logique de comptage originale
    - Ajoute des statistiques utiles
    """
    # Initialisation des compteurs pour TOUS vos labels
    stats = {
        # Entités de base
        'B-COMPANY': [],
        'B-DATE': [],
        'B-ADDRESS': [],
        'B-TOTAL': [],
        
        # Facture
        'B-INVOICE_NO': [],
        'I-INVOICE_NO': [],
        
        # Items
        'B-ITEM_DESC': [],
        'I-ITEM_DESC': [],
        'B-ITEM_QTY': [],
        'I-ITEM_QTY': [],
        'B-ITEM_PRICE': [],
        'I-ITEM_PRICE': [],
        
        # Stats globales
        'not_other': [],
        'total_words': []
    }

    for _, row in df.iterrows():
        tags = row['ner_tags']
        words = row['words']
        
        # Comptage pour chaque label
        counts = {label: 0 for label in stats}
        counts['total_words'] = len(words)
        
        for tag_id in tags:
            label = ids2labels[tag_id]
            if label in counts:
                counts[label] += 1
            if tag_id != labels2ids['O']:
                counts['not_other'] += 1
        
        # Ajout au résultat final
        for label in stats:
            stats[label].append(counts[label])
    
    return pd.DataFrame(stats)

# Utilisation identique à votre demande
#stats_df = create_stats_df(train_df)

# Affichage ciblé comme demandé
#selected_cols = [
#    'B-COMPANY', 'B-DATE', 'B-ADDRESS', 'B-TOTAL',
#    'B-INVOICE_NO', 'B-ITEM_DESC', 'B-ITEM_QTY',
#    'B-ITEM_PRICE', 'not_other', 'total_words'
#]
#stats_df[selected_cols].describe()
#
def update_model_info(new_path, f1, accuracy):
    """Met à jour complètement le fichier info_modal.json avec les nouvelles informations
    
    Args:
        new_path (str): Le nouveau chemin vers le modèle
        f1 (float): Le nouveau score F1
        accuracy (float): La nouvelle accuracy
    """
    new_info = {
        "model_path": new_path,
        "f1_score": f1,
        "accuracy": accuracy,
        "creation_date": datetime.now().strftime("%Y-%m-%d")
    }
    
    with open("info_modal.json", 'w') as f:
        json.dump(new_info, f, indent=4)

#####telechargement de path model
def get_model_path(config_path="info_modal.json"):
    """
    Lit le fichier de configuration et retourne uniquement le chemin du modèle
    
    Returns:
        str: Le chemin absolu vers le modèle
        
    Raises:
        FileNotFoundError: Si le fichier config ou le modèle n'existe pas
        ValueError: Si le fichier config est mal formaté
    """
    # 1. Vérifier que le fichier config existe
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Fichier de configuration introuvable: {config_path}")
    
    # 2. Lire le fichier JSON
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    # 3. Vérifier que la clé model_path existe
    if "model_path" not in config:
        raise ValueError("Le fichier config doit contenir une clé 'model_path'")
    
    model_path = config["model_path"]
    
    # 4. Vérifier que le chemin du modèle existe
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Le chemin du modèle spécifié n'existe pas: {model_path}")
    
    return model_path

def prepare_examples(examples):
  images = [Image.open(path).convert('RGB') for path in examples["image_path"]]
  words = examples["words"]
  bboxes = examples["bboxes"]
  ner_tags = examples["ner_tags"]
  encoding = processor(images, words, boxes=bboxes, word_labels=ner_tags, padding="max_length", truncation=True)
  return encoding
def compute_metrics(p):
  predictions, labels = p
  predictions = np.argmax(predictions, axis=2)
  true_predictions = [
    [labels_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
  ]
  true_labels = [
    [labels_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
  ]
  results = seqeval.compute(predictions=true_predictions, references=true_labels)
  return {
      "precision": results["overall_precision"],
      "recall": results["overall_recall"],
      "f1": results["overall_f1"],
      "accuracy": results["overall_accuracy"],
  }


###############################################
# === Point d'entrée principal ===
if __name__ == "__main__":
    #config
    config = {
    "mongo": {
        "host": "localhost",
        "port": 27017,
        "db": "Telnet",
        "collections": {
            "invoices": "invoices",
            "images": "imageinvoices"
        }
    },
    "paths": {
        "source_images": "C:/Users/HP/back_end_final/uploads2",
   #     "dest_images": os.path.join(output_folder, "img"),
    }
    }
    # Exécution séquentielle
    bases_folder='C:/Users/HP/Desktop/sequencedemodel'
    output_folder = init_environment()
    export_invoices(output_folder)
    export_images(output_folder)
    #process_images_with_ocr(output_folder)
    input_folder=os.path.join(output_folder, "img")
    output_boxfolder=os.path.join(output_folder, "box")
    process_images_folder(input_folder=input_folder, output_folder=output_boxfolder)
 # === Étape : Nettoyage des données ===
    print("\n" + "="*50)
    print(" Nettoyage des fichiers orphelins")
    print("="*50)
    deleted_count = clean_orphaned_entities(output_folder)
    print(f"→ {deleted_count} fichiers orphelins supprimés")
    import os

    data_unprocessed = os.path.join(output_folder, "data/unprocessed")

    # 2. Lister uniquement les triplets complets (image + box + entities existants)
    valid_files = []
    for img_file in os.listdir(os.path.join(output_folder, "img")):
        base_name = os.path.splitext(img_file)[0]
        box_path = os.path.join(output_folder, "box", f"{base_name}.txt")
        entities_path = os.path.join(output_folder, "entities", f"{base_name}.txt")
        
        if os.path.exists(box_path) and os.path.exists(entities_path):
            valid_files.append(base_name)

    # 3. Mélanger et séparer en train/test (70%/30%)
    random.shuffle(valid_files)
    split_idx = int(0.7 * len(valid_files))
    train_files = valid_files[:split_idx]
    test_files = valid_files[split_idx:]
    
    # 4. Copier les triplets complets
    def copy_triplets(file_list, split_name):
        split_dir = os.path.join(data_unprocessed, split_name)
        os.makedirs(os.path.join(split_dir, "img"), exist_ok=True)
        os.makedirs(os.path.join(split_dir, "box"), exist_ok=True)
        os.makedirs(os.path.join(split_dir, "entities"), exist_ok=True)
    
        for base_name in file_list:
            # Copier l'image (supporte .png/.jpg/.jpeg)
            for ext in ['.png', '.jpg', '.jpeg']:
                src_img = os.path.join(output_folder, "img", f"{base_name}{ext}")
                if os.path.exists(src_img):
                    shutil.copy(src_img, os.path.join(split_dir, "img"))
            
            # Copier les annotations
            shutil.copy(
                os.path.join(output_folder, "box", f"{base_name}.txt"),
                os.path.join(split_dir, "box")
            )
            shutil.copy(
                os.path.join(output_folder, "entities", f"{base_name}.txt"),
                os.path.join(split_dir, "entities")
            )
    
    # 5. Exécuter la copie
    copy_triplets(train_files, "train")
    copy_triplets(test_files, "test")
    
    # 6. Vérification
    print(f" Triple cohérents créés dans {data_unprocessed}")
    print(f"Train: {len(train_files)} triplets | Test: {len(test_files)} triplets")
    print(f"Exemple de triplet train : {train_files[0]}.png + {train_files[0]}.txt (box/entities)")    
   
        
    # === 3. Vérification des chemins ===
    print("\nVérification des chemins :")
    print(f"- Train heeeeeeeeeeeeeey : {os.path.join(data_unprocessed, 'train')}")
    print(f"- Test  : {os.path.join(data_unprocessed, 'test')}")
  
    train_df = create_df(os.path.join(data_unprocessed, 'train'))
    test_df = create_df(os.path.join(data_unprocessed, 'test'))
    #train_df = create_df(os.path.join("C:/Users/HP/Desktop/sequencedemodel\03-05-2025\data/unprocessed", 'train'))
    #test_df = create_df(os.path.join("C:/Users/HP/Desktop/sequencedemodel\03-05-2025\data/unprocessed", 'test'))
    len(train_df), len(test_df)
    print(len(train_df), len(test_df))
    ###
    train_stats_df = create_stats_df(train_df)
    train_stats_df.describe()
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)

    print(train_dataset, test_dataset)
    bases_folder = Path('C:/Users/HP/Desktop/sequencedemodel')
    config_path = bases_folder / "modelsfinals" / "config.json.txt"
    
    path=get_model_path(config_path=config_path)
    
    print(path)
    path_to_model = path
    # Load tokenizer and model separately
    tokenizer = AutoTokenizer.from_pretrained(path_to_model,local_files_only=True)
    model = AutoModelForTokenClassification.from_pretrained(path_to_model)
    processor = AutoProcessor.from_pretrained(path_to_model, apply_ocr=False,  local_files_only=True)  # Utilisez path_to_model, pas model_id
    features = Features({
      'pixel_values': Array3D(dtype="float32", shape=(3, 224, 224)),
      'input_ids': Sequence(feature=Value(dtype='int64')),
      'attention_mask': Sequence(Value(dtype='int64')),
      'bbox': Array2D(dtype="int64", shape=(512, 4)),
      'labels': Sequence(feature=Value(dtype='int64')),
    })
    train_dataset = train_dataset.map(prepare_examples, batched=True, remove_columns=train_dataset.column_names, batch_size=32, features=features)
    test_dataset = test_dataset.map(prepare_examples, batched=True, remove_columns=test_dataset.column_names, batch_size=32, features=features)
    #train_dataset.save_to_disk('data/train')
    #test_dataset.save_to_disk('data/test')
    #train_dataset = load_from_disk("data/train")
    #test_dataset = load_from_disk("data/test")
    labels_list = ['O', 'B-COMPANY', 'B-DATE', 'B-ADDRESS', 'B-TOTAL', 'B-INVOICE_NO', 'I-INVOICE_NO',
    'B-ITEM_DESC', 'I-ITEM_DESC',
    'B-ITEM_QTY', 'I-ITEM_QTY',
    'B-ITEM_PRICE', 'I-ITEM_PRICE']
    ids2labels = {k: v for k, v in enumerate(labels_list)}
    labels2ids = {v: k for k, v in enumerate(labels_list)}

    model = AutoModelForTokenClassification.from_pretrained(path_to_model, label2id=labels2ids, id2label=ids2labels)
    seqeval = evaluate.load("seqeval")
    print(train_dataset)
    print(test_dataset)
    from transformers import TrainingArguments

    training_args = TrainingArguments(output_dir="checkpoints",
                                      num_train_epochs=10,
                                      eval_steps=10,
                                      load_best_model_at_end=True,
                                      metric_for_best_model="f1",
                                      per_device_train_batch_size=4,
                                      per_device_eval_batch_size=4,
                                      save_strategy="steps",
                                      eval_strategy="steps",
                                      report_to="none")
    trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=processor,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    )
    trainer.evaluate()
    eval_results = trainer.evaluate()

    # Charger les informations actuelles du modèle
    config_path = bases_folder / "modelsfinals" / "config.json.txt"
    with open(config_path, 'r', encoding='utf-8') as f:
        model_info = json.load(f)
    
    current_f1 = model_info["f1_score"]
    new_f1 = eval_results["eval_f1"]
    
    print(f"\nComparaison des scores F1:")
    print(f"- Ancien score F1: {current_f1:.4f}")
    print(f"- Nouveau score F1: {new_f1:.4f}")
    
    if new_f1 > current_f1:
        # Sauvegarder le nouveau modèle
        new_model_path = bases_folder / "modelsfinals" 
        model.save_pretrained(new_model_path)
        processor.save_pretrained(new_model_path)
        
        # Mettre à jour le fichier info
        update_model_info(
            new_path=str(new_model_path),
            f1=new_f1,
            accuracy=eval_results["eval_accuracy"]
        )
        
        print("\n Le fine-tuning a amélioré le modèle!")
        print(f"Modèle sauvegardé dans: {new_model_path}")
    else:
        print("\n Le fine-tuning n'a pas amélioré le modèle (F1 score égal ou inférieur)")
        print("Conservation du modèle précédent")
    
   
    print("Exécuté avec succès!!!!!")

Image sauvegardée: 6829d8a257a32d3ef204ab65.jpg (original: 1747572873170-processed_invoice_1747572873134.jpg)
Image sauvegardée: 6829da5d57a32d3ef204ac96.jpg (original: 1747573308479-processed_invoice_1747573308054.jpg)
Image sauvegardée: 6829ecb736b47ae76ab57640.jpg (original: 1747578019867-processed_invoice_1747578020155.jpg)
Image sauvegardée: 6829f19a42650487cf7817b6.jpg (original: 1747579250041-processed_invoice_1747579250823.jpg)
Image sauvegardée: 6829f5c4f5574ee82a6c635b.jpg (original: 1747580342379-processed_invoice_1747580343275.jpg)


Processing images:   0%|          | 0/5 [00:00<?, ?it/s]


Chemin absolu du dossier d'images: C:\Users\HP\Desktop\sequencedemodel\27-05-2025\img
Chemin absolu du dossier de sortie: C:\Users\HP\Desktop\sequencedemodel\27-05-2025



Processing images:  20%|██        | 1/5 [00:00<00:01,  3.27it/s]


Chemin absolu du dossier d'images: C:\Users\HP\Desktop\sequencedemodel\27-05-2025\img
Chemin absolu du dossier de sortie: C:\Users\HP\Desktop\sequencedemodel\27-05-2025



...

Processing images: 100%|██████████| 5/5 [00:01<00:00,  3.19it/s]



 Nettoyage des fichiers orphelins
Supprimé : 681cf4d0e9fe46a46530e49f.txt (image manquante)
Supprimé : 6828fcce4e22e2cad532959d.txt (image manquante)
Supprimé : 6828fd034e22e2cad53295bf.txt (image manquante)
Supprimé : 682916c5d705c92813e74b50.txt (image manquante)
Supprimé : 6829f705f5574ee82a6c63c4.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a21.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a23.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a25.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a27.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a29.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a2b.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a2d.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a2f.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a31.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a33.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a35.txt (image manquante)
Supprimé : 6831f89beb644f2ffce24a37.t

Map: 100%|██████████| 2/2 [00:00<00:00, 47.51 examples/s]


Dataset({
    features: ['pixel_values', 'input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 3
})
Dataset({
    features: ['pixel_values', 'input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 2
})



Comparaison des scores F1:
- Ancien score F1: 0.8300
- Nouveau score F1: 0.3333

 Le fine-tuning n'a pas amélioré le modèle (F1 score égal ou inférieur)
Conservation du modèle précédent
Exécuté avec succès!!!!!
